# 🚀 Beginner's Guide to Gradio (Day 12)

Gradio is a powerful Python library that lets you quickly build web-based user interfaces for your machine learning models, data science workflows, or Python functions.

### Why Gradio vs Streamlit?
| Feature | Streamlit (`Day 11`) | Gradio (`Day 12`) |
| :--- | :--- | :--- |
| **Primary Use Case** | Data dashboards, internal tools | ML model demos, quick APIs, interactive inputs |
| **Execution Model** | Reruns entire script from top to bottom on input change | Event-driven (only runs specific function linked to widget/event) |
| **Sharing** | Requires hosting on Streamlit Cloud/server | Built-in `share=True` generates instant 72-hour public link |

---

## 1. Installation & Imports
Run the cell below to ensure Gradio and Pandas are installed and imported.

In [ ]:
import gradio as gr
import pandas as pd
print(f"✅ Gradio version: {gr.__version__}")

## 2. Quick Start: `gr.Interface`
`gr.Interface` is the simplest way to wrap any Python function with a UI.
You just specify:
1. `fn`: The Python function to run.
2. `inputs`: List of Gradio input components (or strings like `'textbox'`, `'slider'`).
3. `outputs`: List of Gradio output components.

In [ ]:
def greet(name, intensity):
    if not name:
        name = "World"
    return f"Hello, {name}! " + "🎉" * int(intensity)

quick_demo = gr.Interface(
    fn=greet,
    inputs=[gr.Textbox(label="Your Name"), gr.Slider(minimum=1, maximum=5, step=1, label="Intensity")],
    outputs=gr.Textbox(label="Greeting"),
    title="Quick Interface Example",
    description="Type your name and adjust the slider to see Gradio in action!"
)

# Launch in notebook cell
quick_demo.launch(inline=True)

## 3. Dynamic User Input Form with `gr.Blocks`
To build custom layouts (like rows, columns, tabs, or multi-step forms just like we did in `Day11(streamlit basics).py`), we use `gr.Blocks()`.

Below, we map every input widget from `Day 11 Streamlit` to its exact `Gradio` equivalent:

In [ ]:
def process_form(name, age, gender, education, skills, experience, resume, relocate):
    skills_str = ", ".join(skills) if skills else "None selected"
    
    result = {
        "Name": name if name else "Not provided",
        "Age": int(age) if age is not None else 18,
        "Gender": gender if gender else "Not specified",
        "Education": education if education else "Not specified",
        "Skills": skills_str,
        "Experience": int(experience),
        "Relocate": "Yes" if relocate else "No"
    }
    
    df = pd.DataFrame([result])
    
    if resume is not None:
        file_name = resume if isinstance(resume, str) else resume.name
        file_status = f"✅ Uploaded File: {file_name}"
    else:
        file_status = "ℹ️ No resume uploaded yet."
        
    return result, df, file_status


with gr.Blocks(title="Day 12 Form Demo", theme=gr.themes.Soft()) as form_demo:
    gr.Markdown("## 📝 Dynamic User Input Form (`gr.Blocks` Example)")
    gr.Markdown("Fill out the form below and click **Submit Form** to see the dictionary and Pandas DataFrame outputs.")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Personal Information")
            form_name = gr.Textbox(label="Enter your Name", placeholder="Full Name")
            form_age = gr.Number(label="Enter Age", value=18, minimum=1, maximum=100)
            form_gender = gr.Radio(choices=["Male", "Female", "Other"], label="Select Gender", value="Male")
            form_education = gr.Dropdown(
                choices=["High School", "Diploma", "Bachelor's", "Master's", "PhD"],
                label="Highest Qualification",
                value="Bachelor's"
            )
            form_skills = gr.CheckboxGroup(
                choices=["Python", "Java", "SQL", "Machine Learning", "Deep Learning", "Gradio"],
                label="Select Skills"
            )
            form_experience = gr.Slider(minimum=0, maximum=20, value=1, step=1, label="Years of Experience")
            form_resume = gr.File(label="Upload Resume", file_types=[".pdf", ".docx"])
            form_relocate = gr.Checkbox(label="Willing to Relocate", value=False)
            
            submit_btn = gr.Button("Submit Form", variant="primary")
            
        with gr.Column(scale=1):
            gr.Markdown("### Collected Results")
            file_status_output = gr.Textbox(label="File Upload Status", interactive=False)
            json_output = gr.JSON(label="Result as JSON (Dictionary)")
            dataframe_output = gr.Dataframe(label="Result as Pandas DataFrame", interactive=False)
            
    submit_btn.click(
        fn=process_form,
        inputs=[form_name, form_age, form_gender, form_education, form_skills, form_experience, form_resume, form_relocate],
        outputs=[json_output, dataframe_output, file_status_output]
    )

form_demo.launch(inline=True, share=True)

## 4. Live Interactivity (Event Listeners)
Unlike Streamlit, Gradio allows you to trigger calculations **live** as any widget changes using `.change()` events.

In [ ]:
def calculate(num1, num2, operation):
    if operation == "Add (+)":
        return num1 + num2
    elif operation == "Subtract (-)":
        return num1 - num2
    elif operation == "Multiply (×)":
        return num1 * num2
    elif operation == "Divide (÷)":
        return num1 / num2 if num2 != 0 else "Error: Division by Zero"
    return 0

with gr.Blocks() as calc_demo:
    gr.Markdown("### Live Calculator Demo")
    with gr.Row():
        with gr.Column():
            n1 = gr.Number(label="Number 1", value=10)
            n2 = gr.Number(label="Number 2", value=5)
            op = gr.Radio(["Add (+)", "Subtract (-)", "Multiply (×)", "Divide (÷)"], label="Operation", value="Add (+)")
        with gr.Column():
            ans = gr.Textbox(label="Calculated Result", interactive=False)
            
    for comp in [n1, n2, op]:
        comp.change(fn=calculate, inputs=[n1, n2, op], outputs=ans)

calc_demo.launch(inline=True)